# Maia Lite Chat 0.3.0 — Corpus Acquisition v8.7

Resumable Hugging Face streaming cursors, retained in-memory iterators across rounds, and a 22-hour graceful session limit. **Uses the same RUN_ID and committed corpus directory as v8.6.**

**Important:** Stop the old notebook before running this version. Execute cells in order. Cursors are optimizations; committed chunks remain authoritative. The first run may replay previously scanned data before reaching new documents.


# Maia Corpus v8.9 — instrumented acquisition diagnostics

Based on v8.8 safe recovery. The acquisition remains **OFF by default**. Diagnostic events are timestamped and appended to `reports/acquisition_diagnostics_v8_9.jsonl`. A 30-second heartbeat reports which operation is blocking (not a timeout). Committed chunks, cursors, quotas and RUN_ID are unchanged.

**Run setup, metadata, definitions and recovery first.** To diagnose the freeze, explicitly enable acquisition in the final cell. Do not run acquisition simultaneously in another Colab session.


In [ ]:
%pip -q install "datasets>=3.0,<5" "huggingface_hub>=0.26" "pandas>=2.0" "tqdm>=4.60"

In [ ]:
import os, json, gzip, hashlib, re, random, time, shutil, sqlite3, math, pickle
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import pandas as pd
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names
from huggingface_hub import HfApi

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; configure ROOT manually if needed.')

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
RUN_ID = 'acquisition_v8_5_001'  # REUSE validated 100M; do not change without intention
RUN_DIR = ROOT / 'corpus_v8' / 'runs' / RUN_ID
DRIVE_CANDIDATES = RUN_DIR / 'quarantine_pending_rights_review'
REPORTS = RUN_DIR / 'reports'
DRIVE_CANDIDATES.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# Set to True to write new chunks on the ephemeral Colab SSD and upload to Drive
# after each atomic chunk. Never count an unsynced local chunk as committed.
USE_LOCAL_SSD_STAGING = True
LOCAL_STAGE = Path('/content/maia_corpus_staging')
LOCAL_STAGE.mkdir(parents=True, exist_ok=True)
CANDIDATES = DRIVE_CANDIDATES  # committed truth; do not store permanent data only on SSD

DRY_RUN = False  # v8.7: acquisition enabled; set True to inspect only
TARGET_PROXY_TOKENS = 8_200_000_000  # provisional; recalibrate with actual tokenizer
SEED = 20261018
SHUFFLE_BUFFER = 1000
MAX_SECONDS_PER_JOB = 3600  # maximum duration per source attempt
MAX_ROWS_SCANNED_PER_JOB = 20_000_000
MAX_TEXT_CHARS = 100_000
MIN_TEXT_CHARS = 250
CHUNK_DOCS = 500
MAX_DRIVE_GB = 250.0  # guardrail, not prediction; check Drive quota first
MIN_FREE_LOCAL_GB = 5.0
RUN_STREAMING = True
MAX_SESSION_HOURS = 22.0  # stop gracefully before typical Colab session expiry
SESSION_STARTED_MONOTONIC = time.monotonic()
SESSION_DEADLINE = SESSION_STARTED_MONOTONIC + MAX_SESSION_HOURS * 3600
CURSOR_DIR = RUN_DIR / "stream_cursors_v8_7"
CURSOR_DIR.mkdir(parents=True, exist_ok=True)
# Continuous acquisition: run until quotas are reached or a real blocker occurs.
# A round-robin slice is not a session cap; the loop continues automatically.
ACQUISITION_SLICE_PROXY_TOKENS = 5_000_000
MAX_CONSECUTIVE_STALLED_ROUNDS = 1
MAX_NEW_PROXY_TOKENS_PER_SESSION = None  # intentionally unlimited
MAX_NEW_PROXY_TOKENS_PER_JOB = None      # intentionally unlimited

assert TARGET_PROXY_TOKENS > 0
assert ACQUISITION_SLICE_PROXY_TOKENS > 0
print('Run:', RUN_DIR, 'Dry run:', DRY_RUN)
print('Existing corpus will be reused; no overwrite of committed chunks.')


## 1. Six explicit acquisition jobs

The 100M pilot uses **web 80% / Wikipedia 20%** to fill the available six language/source cells. This is deliberately different from a proposed final corpus mixture; it does not pretend that dialogue/logic/creative sources are ready.

**Important:** the target is a *ceiling* for candidate data, not a promise that the remote services can deliver all of it within runtime and storage limits.


In [ ]:
LANG_SHARES = {'pt': 0.40, 'en': 0.35, 'es': 0.25}
FAMILY_SHARES = {'web': 0.80, 'wikipedia': 0.20}
JOBS = [
    {'id':'fineweb2_pt','dataset':'HuggingFaceFW/fineweb-2','config':'por_Latn','lang':'pt','family':'web'},
    {'id':'fineweb2_es','dataset':'HuggingFaceFW/fineweb-2','config':'spa_Latn','lang':'es','family':'web'},
    {'id':'fineweb_edu_en','dataset':'HuggingFaceFW/fineweb-edu','config':'sample-10BT','lang':'en','family':'web'},
    {'id':'wikipedia_pt','dataset':'wikimedia/wikipedia','config':'20231101.pt','lang':'pt','family':'wikipedia'},
    {'id':'wikipedia_en','dataset':'wikimedia/wikipedia','config':'20231101.en','lang':'en','family':'wikipedia'},
    {'id':'wikipedia_es','dataset':'wikimedia/wikipedia','config':'20231101.es','lang':'es','family':'wikipedia'},
]
for job in JOBS:
    job['target_proxy_tokens'] = round(TARGET_PROXY_TOKENS * LANG_SHARES[job['lang']] * FAMILY_SHARES[job['family']])
print(pd.DataFrame(JOBS)[['id','lang','family','target_proxy_tokens']].to_string(index=False))
assert sum(j['target_proxy_tokens'] for j in JOBS) == TARGET_PROXY_TOKENS


## 2. Verify Hub metadata, configurations and split availability

Do **not** pass `trust_remote_code` to `get_dataset_config_names` or `get_dataset_split_names`; it causes compatibility errors in the Colab `datasets` environment seen in v8.4.


In [ ]:
api = HfApi()
source_metadata = {}
job_checks = []
for dataset_id in sorted(set(j['dataset'] for j in JOBS)):
    entry = {'dataset': dataset_id, 'sha': None, 'license_claim': None, 'configs': [], 'error': None}
    try:
        info = api.dataset_info(dataset_id)
        entry['sha'] = info.sha
        entry['license_claim'] = (info.cardData or {}).get('license')
        entry['gated'] = info.gated
    except Exception as e:
        entry['error'] = f'Hub metadata: {type(e).__name__}: {str(e)[:250]}'
    try:
        entry['configs'] = get_dataset_config_names(dataset_id)
    except Exception as e:
        entry['error'] = (entry['error'] or '') + f' Config discovery: {type(e).__name__}: {str(e)[:250]}'
    source_metadata[dataset_id] = entry
for job in JOBS:
    entry = source_metadata[job['dataset']]
    valid = job['config'] in entry['configs']
    check = {'job_id': job['id'], 'config_valid': valid, 'split': None, 'error': None}
    if valid:
        try:
            splits = get_dataset_split_names(job['dataset'], config_name=job['config'])
            check['split'] = 'train' if 'train' in splits else (splits[0] if splits else None)
            if check['split'] is None: check['error'] = 'No split available'
        except Exception as e:
            check['error'] = f'{type(e).__name__}: {str(e)[:250]}'
    else:
        check['error'] = f'Config not available: {job["config"]}'
    job_checks.append(check)
    print(job['id'], 'config:', valid, 'split:', check['split'], 'error:', check['error'])
check_by_id = {c['job_id']: c for c in job_checks}
(REPORTS / 'source_metadata.json').write_text(json.dumps(source_metadata, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
(REPORTS / 'job_checks.json').write_text(json.dumps(job_checks, indent=2, ensure_ascii=False), encoding='utf-8')


## 3. Crash-safe committed chunks and resumable deduplication

Each compressed chunk is written to a temporary file, then atomically renamed to a final filename. On restart, we rebuild the exact-deduplication index and counters from **committed final chunks**, so interrupted writes do not create phantom progress. Chunk sizes are deliberately small for Drive.

A dataset's streaming iterator may restart from the beginning when Colab reconnects; exact deduplication prevents duplicate storage, but rescanning can consume time. A checkpointed dataset iterator is **not** claimed.


In [ ]:
def utcnow():
    return datetime.now(timezone.utc).isoformat()

def normalize_text(text):
    return re.sub(r'\s+', ' ', text.casefold()).strip()

def sha256_text(text):
    return hashlib.sha256(normalize_text(text).encode('utf-8')).hexdigest()

def token_proxy(text):
    return max(1, round(len(text)/4))

def extract_text(row):
    for key in ('text', 'content', 'body', 'document', 'raw_content'):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value, key
    return None, None

def committed_chunks(job_id):
    return sorted(DRIVE_CANDIDATES.glob(f'{job_id}_chunk_*.jsonl.gz'))

def reconstruct_state():
    hashes = set()
    stats = {j['id']: {'stored_docs':0,'proxy_tokens':0,'bytes_uncompressed':0,'chunks':0} for j in JOBS}
    corrupt = []
    for job in JOBS:
        for file in committed_chunks(job['id']):
            try:
                with gzip.open(file, 'rt', encoding='utf-8') as handle:
                    for line in handle:
                        row = json.loads(line)
                        key = row['normalized_sha256']
                        if key in hashes:
                            raise ValueError('Duplicate hash across committed chunks: ' + key)
                        hashes.add(key)
                        stats[job['id']]['stored_docs'] += 1
                        stats[job['id']]['proxy_tokens'] += int(row['proxy_tokens'])
                        stats[job['id']]['bytes_uncompressed'] += int(row['text_bytes'])
                stats[job['id']]['chunks'] += 1
            except Exception as e:
                corrupt.append({'file':str(file),'error':f'{type(e).__name__}: {str(e)[:200]}'})
    if corrupt:
        raise RuntimeError('Corrupt/duplicate committed chunk(s); do not continue: ' + json.dumps(corrupt[:5]))
    return hashes, stats

## Continuous acquisition

This version runs successive small, interleaved source slices **without a 100M-per-execution cap**. It stops when all six quotas are satisfied or when a full round produces no documents. Real storage and runtime limits remain safety guards. After Colab disconnects, rerun the notebook to reconstruct committed state. Existing 200M proxy tokens are preserved.

In [ ]:
# v8.9 diagnostics: time-stamped events and a heartbeat during blocking calls.
# The heartbeat only reads diagnostic state; it never touches the HF iterator.
import threading
DIAG = {'phase': 'idle', 'job': None, 'since': time.monotonic(), 'rows': 0}
DIAG_LOG = REPORTS / 'acquisition_diagnostics_v8_9.jsonl'
DIAG_LOCK = threading.Lock()

def diag_event(event, job=None, **details):
    payload = {'utc': utcnow(), 'event': event, 'job': job, **details}
    print('[DIAG]', json.dumps(payload, ensure_ascii=False, default=str), flush=True)
    try:
        with DIAG_LOCK:
            with DIAG_LOG.open('a', encoding='utf-8') as f:
                f.write(json.dumps(payload, ensure_ascii=False, default=str) + '\n')
    except Exception as exc:
        print('[DIAG] Could not write log:', repr(exc), flush=True)

def diag_phase(phase, job=None):
    DIAG.update(phase=phase, job=job, since=time.monotonic())

def diag_heartbeat(stop_event, interval=30):
    while not stop_event.wait(interval):
        diag_event('heartbeat', DIAG.get('job'), phase=DIAG.get('phase'),
                   elapsed_seconds=round(time.monotonic()-DIAG.get('since',time.monotonic()),1),
                   rows=DIAG.get('rows'))

def quarantine_size_bytes():
    return sum(f.stat().st_size for f in DRIVE_CANDIDATES.glob('*.jsonl.gz'))

def disk_ok():
    try:
        free = shutil.disk_usage('/content').free
        return free >= MIN_FREE_LOCAL_GB * (1024**3)
    except Exception:
        return True

def commit_chunk(job_id, batch, chunk_index):
    if not batch:
        return None
    final = DRIVE_CANDIDATES / f'{job_id}_chunk_{chunk_index:06d}.jsonl.gz'
    if final.exists():
        raise FileExistsError(f'Committed chunk already exists: {final}')
    work_dir = LOCAL_STAGE if USE_LOCAL_SSD_STAGING else DRIVE_CANDIDATES
    tmp = work_dir / f'.{job_id}_chunk_{chunk_index:06d}.{os.getpid()}.tmp'
    transfer = DRIVE_CANDIDATES / f'.{job_id}_chunk_{chunk_index:06d}.{os.getpid()}.uploading'
    try:
        with gzip.open(tmp, 'wt', encoding='utf-8', compresslevel=5) as handle:
            for record in batch:
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        # Hash of compressed bytes, verified after copying to Drive.
        def file_sha256(path):
            h = hashlib.sha256()
            with open(path, 'rb') as f:
                for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
                    h.update(block)
            return h.hexdigest()
        digest = file_sha256(tmp)
        shutil.copyfile(tmp, transfer)
        if file_sha256(transfer) != digest:
            raise IOError('Drive copy hash mismatch: ' + str(transfer))
        # Verify gzip stream and JSON before commit.
        with gzip.open(transfer, 'rt', encoding='utf-8') as handle:
            verified = sum(1 for line in handle if json.loads(line))
        if verified != len(batch):
            raise IOError(f'Unexpected row count: {verified} != {len(batch)}')
        os.replace(transfer, final)
        return final
    finally:
        if tmp.exists(): tmp.unlink()
        if transfer.exists(): transfer.unlink()

# v8.7: a streaming iterator per job is reused across all acquisition rounds.
# Dataset state_dict checkpoints are written ONLY after committed chunk flush.
# A cursor is an optimization, not the source of truth: committed JSONL chunks are.
ACTIVE_STREAMS = {}
ACTIVE_ITERATORS = {}

def cursor_file(jid):
    return CURSOR_DIR / f'{jid}.pkl'

def make_stream(job, check, stats):
    jid = job['id']
    if jid in ACTIVE_STREAMS:
        return ACTIVE_ITERATORS[jid]
    diag_phase('load_dataset', jid)
    diag_event('load_dataset_start', jid)
    _t=time.monotonic()
    ds = load_dataset(job['dataset'], name=job['config'], split=check['split'], streaming=True)
    diag_event('load_dataset_done', jid, seconds=round(time.monotonic()-_t,2))
    diag_phase('shuffle_setup', jid)
    _t=time.monotonic()
    ds = ds.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
    diag_event('shuffle_setup_done', jid, seconds=round(time.monotonic()-_t,2), buffer=SHUFFLE_BUFFER)
    path = cursor_file(jid)
    if path.exists():
        try:
            # Only read checkpoints produced by this notebook in your own Drive.
            with open(path, 'rb') as f:
                checkpoint = pickle.load(f)
            if (checkpoint.get('job_id') != jid or checkpoint.get('seed') != SEED
                or checkpoint.get('dataset') != job['dataset']
                or checkpoint.get('config') != job['config']
                or checkpoint.get('split') != check['split']
                or checkpoint.get('source_revision') != source_metadata[job['dataset']]['sha']):
                raise ValueError('Cursor metadata changed; restart stream with deduplication')
            if checkpoint.get('committed_tokens', 0) > stats[jid]['proxy_tokens']:
                raise ValueError('Cursor ahead of committed data; refusing unsafe resume')
            diag_phase('cursor_load_state_dict', jid)
            _t=time.monotonic()
            ds.load_state_dict(checkpoint['state'])
            diag_event('cursor_load_state_dict_done', jid, seconds=round(time.monotonic()-_t,2), saved_tokens=checkpoint.get('committed_tokens'), committed_tokens=stats[jid]['proxy_tokens'])
            print(f'Cursor restored: {jid}; saved tokens={checkpoint["committed_tokens"]:,}', flush=True)
        except Exception as exc:
            print(f'WARNING: cursor restore failed for {jid}: {exc}. Starting stream from beginning; dedup stays active.', flush=True)
    ACTIVE_STREAMS[jid] = ds
    diag_phase('iterator_creation', jid)
    _t=time.monotonic()
    ACTIVE_ITERATORS[jid] = iter(ds)
    diag_event('iterator_created', jid, seconds=round(time.monotonic()-_t,2))
    return ACTIVE_ITERATORS[jid]

def save_stream_cursor(job, check, stats):
    jid = job['id']
    if jid not in ACTIVE_STREAMS:
        return
    checkpoint = {
        'format': 'maia-acquisition-stream-v8.7', 'job_id': jid,
        'dataset': job['dataset'], 'config': job['config'], 'split': check['split'],
        'source_revision': source_metadata[job['dataset']]['sha'],
        'seed': SEED, 'committed_tokens': stats[jid]['proxy_tokens'],
        'committed_docs': stats[jid]['stored_docs'],
        'state': ACTIVE_STREAMS[jid].state_dict(), 'saved_utc': utcnow()
    }
    path = cursor_file(jid)
    tmp = LOCAL_STAGE / f'.{jid}.cursor.{os.getpid()}.tmp'
    transfer = CURSOR_DIR / f'.{jid}.cursor.{os.getpid()}.uploading'
    try:
        with open(tmp, 'wb') as f:
            pickle.dump(checkpoint, f, protocol=4)
        shutil.copyfile(tmp, transfer)
        os.replace(transfer, path)
    finally:
        tmp.unlink(missing_ok=True)
        transfer.unlink(missing_ok=True)

def acquire_job(job, check, global_hashes, stats, slice_budget):
    jid = job['id']
    report = {'job_id':jid,'status':'not_run','rows_scanned':0,'accepted_docs_this_run':0,
              'new_proxy_tokens':0,'duplicate_rows':0,'short_rows':0,'error':None,
              'target_proxy_tokens':job['target_proxy_tokens'], 'started_utc':utcnow()}
    if DRY_RUN or not RUN_STREAMING:
        report['status'] = 'dry_run'; return report
    if check['error'] or not check['config_valid'] or not check['split']:
        report['status']='config_error';report['error']=check['error'];return report
    if time.monotonic() >= SESSION_DEADLINE:
        report['status']='session_deadline';return report
    if slice_budget <= 0:
        report['status']='slice_complete';return report
    if stats[jid]['proxy_tokens'] >= job['target_proxy_tokens']:
        report['status']='quota_met_previous_run';return report
    batch = []
    pending_hashes = set()
    next_chunk = max([int(f.stem.split('_')[-1].split('.')[0]) for f in committed_chunks(jid)], default=-1)+1
    # .stem for *.jsonl.gz ends in '.jsonl'; parsed index is split('.')[0].
    started=time.monotonic()
    diag_phase('make_stream', jid)
    diag_event('job_start', jid, slice_budget=slice_budget, committed_tokens=stats[jid]['proxy_tokens'])
    per_job_start_tokens=stats[jid]['proxy_tokens']
    try:
        iterator = make_stream(job, check, stats)
        index = 0
        while True:
            if time.monotonic() >= SESSION_DEADLINE:
                report['status']='session_deadline';break
            if stats[jid]['proxy_tokens'] + sum(x['proxy_tokens'] for x in batch) - per_job_start_tokens >= slice_budget:
                report['status']='slice_complete';break
            if stats[jid]['proxy_tokens'] + sum(x['proxy_tokens'] for x in batch) >= job['target_proxy_tokens']:
                report['status']='quota_met';break
            try:
                diag_phase('next_iterator', jid)
                _read_start = time.monotonic()
                row = next(iterator)
                _read_seconds = time.monotonic() - _read_start
                if _read_seconds >= 5 or index == 0:
                    diag_event('next_iterator_returned', jid, seconds=round(_read_seconds,2), rows_scanned=report['rows_scanned'])
            except StopIteration:
                report['status']='source_exhausted';break
            index += 1
            if index >= MAX_ROWS_SCANNED_PER_JOB:
                report['status']='row_budget_reached';break
            if time.monotonic()-started >= MAX_SECONDS_PER_JOB:
                report['status']='time_budget_reached';break
            if index % 500 == 0 and quarantine_size_bytes() >= MAX_DRIVE_GB * 1024**3:
                report['status']='storage_budget_reached';break
            if index % 500 == 0 and not disk_ok():
                report['status']='local_disk_low';break
            report['rows_scanned'] += 1
            DIAG['rows'] = report['rows_scanned']
            if report['rows_scanned'] % 1000 == 0:
                diag_event('rows_progress', jid, rows_scanned=report['rows_scanned'], accepted_docs=report['accepted_docs_this_run'], duplicates=report['duplicate_rows'], elapsed_seconds=round(time.monotonic()-started,1))
            text, field = extract_text(row)
            if not text or len(text)<MIN_TEXT_CHARS:
                report['short_rows'] += 1;continue
            text = text[:MAX_TEXT_CHARS]
            h=sha256_text(text)
            if h in global_hashes or h in pending_hashes:
                report['duplicate_rows'] += 1;continue
            proxy=token_proxy(text)
            current=stats[jid]['proxy_tokens']+sum(x['proxy_tokens'] for x in batch)
            if current - per_job_start_tokens >= slice_budget:
                report['status']='slice_complete';break
            if current >= job['target_proxy_tokens']:
                report['status']='quota_met';break
            # Last document can exceed quota; recorded as such in manifest.
            doc_id = row.get('id') or row.get('article_id') or row.get('page_id')
            url = row.get('url') or row.get('source_url')
            record={
                'dataset_id':job['dataset'],'dataset_revision':source_metadata[job['dataset']]['sha'],
                'config':job['config'],'split':check['split'],'job_id':jid,
                'source_family':job['family'],'expected_language':job['lang'],
                'source_document_id':str(doc_id) if doc_id is not None else None,
                'source_url':str(url)[:1000] if url is not None else None,
                'text_field':field,'text':text,'text_bytes':len(text.encode('utf-8')),
                'normalized_sha256':h,'proxy_tokens':proxy,
                'rights_status':'pending_review','eligible_for_training':False,
                'acquired_utc':utcnow()
            }
            batch.append(record);pending_hashes.add(h)
            if len(batch)>=CHUNK_DOCS:
                diag_phase('commit_chunk', jid)
                _commit_start=time.monotonic()
                final=commit_chunk(jid,batch,next_chunk)
                diag_event('chunk_committed', jid, seconds=round(time.monotonic()-_commit_start,2), file=final.name)
                for r in batch:global_hashes.add(r['normalized_sha256'])
                stats[jid]['stored_docs']+=len(batch)
                stats[jid]['proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
                stats[jid]['bytes_uncompressed']+=sum(r['text_bytes'] for r in batch)
                stats[jid]['chunks']+=1
                report['accepted_docs_this_run']+=len(batch)
                report['new_proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
                next_chunk+=1;batch=[];pending_hashes.clear()
        else:
            report['status']='source_exhausted'
        if report['status']=='not_run':report['status']='stopped'
    except Exception as e:
        report['status']='error';report['error']=f'{type(e).__name__}: {str(e)[:500]}'
    finally:
        # Commit buffered valid records even when a network exception interrupts the stream.
        if batch:
            try:
                diag_phase('final_commit_chunk', jid)
                _commit_start=time.monotonic()
                commit_chunk(jid,batch,next_chunk)
                diag_event('final_chunk_committed', jid, seconds=round(time.monotonic()-_commit_start,2))
                for r in batch:global_hashes.add(r['normalized_sha256'])
                stats[jid]['stored_docs']+=len(batch)
                stats[jid]['proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
                stats[jid]['bytes_uncompressed']+=sum(r['text_bytes'] for r in batch)
                stats[jid]['chunks']+=1
                report['accepted_docs_this_run']+=len(batch)
                report['new_proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
            except Exception as e:
                report['status']='commit_error';report['error']=f'{type(e).__name__}: {str(e)[:500]}'
        if report['status'] != 'commit_error' and jid in ACTIVE_STREAMS:
            try:
                diag_phase('save_stream_cursor', jid)
                _cursor_start=time.monotonic()
                save_stream_cursor(job, check, stats)
                diag_event('cursor_saved', jid, seconds=round(time.monotonic()-_cursor_start,2))
            except Exception as exc:
                # No data loss: next session replays and deduplicates.
                print(f'WARNING: could not save cursor for {jid}: {exc}', flush=True)
        report['finished_utc']=utcnow()
        diag_phase('idle', jid)
        diag_event('job_finished', jid, status=report['status'], rows=report['rows_scanned'], new_proxy_tokens=report['new_proxy_tokens'], error=report['error'])
    return report



## 4. Recover persisted state (read-only)

This scans the committed gzip chunks and validates exact hashes. It does **not** start acquisition or modify committed chunks. Allow time and RAM for millions of hashes.


In [ ]:
if 'known_hashes' not in globals() or 'committed_stats' not in globals():
    print('Reconstructing committed chunks (read-only)...', flush=True)
    known_hashes, committed_stats = reconstruct_state()
else:
    print('Using reconstructed state already in memory.')
actual_total = sum(s['proxy_tokens'] for s in committed_stats.values())
print('Committed documents:', f'{len(known_hashes):,}')
print('Committed proxy tokens:', f'{actual_total:,}')
print('Remaining:', f'{max(0, TARGET_PROXY_TOKENS-actual_total):,}')
print('Progress:', f'{100*actual_total/TARGET_PROXY_TOKENS:.2f}%')
for j in JOBS:
    s = committed_stats[j['id']]
    print(j['id'], 'stored', f"{s['proxy_tokens']:,}", 'remaining', f"{max(0,j['target_proxy_tokens']-s['proxy_tokens']):,}")


## 5. Reconstruct state and write auditable manifest

The final manifest always reports **candidate** counts, not training-eligible counts. The rights, quality, benchmark and tokenizer gates remain closed. The full 100M acquisition is considered complete only if all six quotas are reached.


In [ ]:
if "known_hashes" not in globals() or "committed_stats" not in globals():
    print("Reconstructing committed corpus; this may take several minutes...", flush=True)
    known_hashes, committed_stats = reconstruct_state()
if "job_reports" not in globals():
    prior_reports = REPORTS / 'last_job_reports.json'
    job_reports = json.loads(prior_reports.read_text(encoding='utf-8')) if prior_reports.exists() else []
actual_total=sum(x['proxy_tokens'] for x in committed_stats.values())
quota_rows=[]
for job in JOBS:
    s=committed_stats[job['id']]
    quota_rows.append({
        'job_id':job['id'],'language':job['lang'],'family':job['family'],
        'target_proxy_tokens':job['target_proxy_tokens'],
        'stored_proxy_tokens':s['proxy_tokens'],
        'remaining_proxy_tokens':max(0,job['target_proxy_tokens']-s['proxy_tokens']),
        'stored_docs':s['stored_docs'],'chunks':s['chunks']
    })
quota_df=pd.DataFrame(quota_rows)
display(quota_df)
quota_df.to_csv(REPORTS/'quota_progress.csv',index=False)
manifest={
    'version':'v8.8-safe-recovery','generated_utc':utcnow(),'run_id':RUN_ID,
    'dry_run':DRY_RUN,
    'max_new_proxy_tokens_per_session':MAX_NEW_PROXY_TOKENS_PER_SESSION,
    'session_limit_hours':MAX_SESSION_HOURS,'cursor_dir':str(CURSOR_DIR),'local_ssd_staging':USE_LOCAL_SSD_STAGING,'target_proxy_tokens':TARGET_PROXY_TOKENS,
    'observed_candidate_proxy_tokens':actual_total,
    'candidate_docs':len(known_hashes),
    'all_quotas_met':all(r['remaining_proxy_tokens']==0 for r in quota_rows),
    'language_target_shares':LANG_SHARES,'acquisition_family_shares':FAMILY_SHARES,
    'quotas':quota_rows,'job_reports':job_reports,
    'quarantine_storage_bytes':quarantine_size_bytes(),
    'sample_method':'seeded streaming shard/buffer shuffle; not dataset-wide probability sample',
    'dedup_scope':'normalized SHA256 exact only across committed v8.5 documents',
    'token_estimator':'round(characters/4), not Maia tokenizer',
    'source_metadata_file':str(REPORTS/'source_metadata.json'),
    'rights_status':'pending_review','eligible_training_tokens':0,
    'production_ready':False,
    'blockers':[
        'Human source/document license and rights review not completed',
        'No final tokenizer; proxy tokens are not actual model tokens',
        'Near-duplicate and multilingual quality audits not performed at full scale',
        'No frozen benchmark contamination audit',
        'Dialogue, logic, creative and other source families still missing',
        'Shard/buffer shuffle is not statistically representative'
    ],
    'outputs':['reports/source_metadata.json','reports/job_checks.json','reports/quota_progress.csv',
               'reports/last_job_reports.json','reports/manifest_v8_8_safe_recovery.json',
               'quarantine_pending_rights_review/*_chunk_*.jsonl.gz']
}
manifest_path=REPORTS/'manifest_v8_8_safe_recovery.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Manifest:',manifest_path)
print('Candidate proxy tokens:',actual_total)
print('All quotas met:',manifest['all_quotas_met'])
print('Production ready:',manifest['production_ready'])


## Optional instrumented acquisition

Set `START_ACQUISITION = True` only when ready to continue acquiring. During a slow `next(iterator)`, a heartbeat prints the current phase every 30 seconds; it does **not** forcibly cancel blocked network operations. Logs go to Drive.


In [ ]:
# EXPLICIT OPT-IN: change to True only after reviewing quotas and source availability.
START_ACQUISITION = False
if not START_ACQUISITION:
    print('Acquisition DISABLED. No source streamed and no chunks written.')
else:
    if 'known_hashes' not in globals() or 'committed_stats' not in globals():
        raise RuntimeError('Run read-only reconstruction first.')
    if 'check_by_id' not in globals() or 'source_metadata' not in globals():
        raise RuntimeError('Run Hub metadata checks first.')
    if DRY_RUN or not RUN_STREAMING:
        raise RuntimeError('Acquisition is disabled by DRY_RUN or RUN_STREAMING.')
    SESSION_STARTED_MONOTONIC = time.monotonic()
    SESSION_DEADLINE = SESSION_STARTED_MONOTONIC + MAX_SESSION_HOURS * 3600
    # Continuous round-robin acquisition. Every source receives repeated opportunities
    # until all quotas are met or no source makes progress in a full round.
    _diag_stop = threading.Event()
    _diag_thread = threading.Thread(target=diag_heartbeat, args=(_diag_stop, 30), daemon=True)
    _diag_thread.start()
    diag_event('acquisition_started', target_proxy_tokens=TARGET_PROXY_TOKENS)
    job_reports=[]
    round_index=0
    exhausted_jobs = set()
    while True:
        if time.monotonic() >= SESSION_DEADLINE:
            print('SESSION DEADLINE: all completed slices committed; stop safely.', flush=True)
            break
        remaining = {j['id']: max(0, j['target_proxy_tokens'] - committed_stats[j['id']]['proxy_tokens']) for j in JOBS}
        if not any(remaining.values()):
            print('All source quotas reached.')
            break
        round_index += 1
        round_new = 0
        print(f'\n=== Acquisition round {round_index} | remaining proxy tokens: {sum(remaining.values()):,} ===', flush=True)
        for job in JOBS:
            if time.monotonic() >= SESSION_DEADLINE:
                print('SESSION DEADLINE: ending round safely.', flush=True)
                break
            jid = job['id']
            if remaining[jid] <= 0 or jid in exhausted_jobs:
                continue
            # Prioritize the most underfilled sources across rounds; each round
            # targets the same small slice per source, rather than stopping at 100M.
            r = acquire_job(job, check_by_id[jid], known_hashes, committed_stats,
                            min(ACQUISITION_SLICE_PROXY_TOKENS, remaining[jid]))
            if r['status'] == 'source_exhausted':
                exhausted_jobs.add(jid)
            r['round'] = round_index
            job_reports.append(r)
            round_new += r['new_proxy_tokens']
            print(jid, r['status'], 'added', f"{r['new_proxy_tokens']:,}",
                  'total', f"{committed_stats[jid]['proxy_tokens']:,}",
                  'error', r['error'], flush=True)
            (REPORTS/'last_job_reports.json').write_text(
                json.dumps(job_reports[-len(JOBS)*3:], indent=2, ensure_ascii=False), encoding='utf-8')
            if r['status'] == 'session_deadline':
                print('SESSION DEADLINE reached; ending acquisition after committed flush.', flush=True)
                break
            if r['status'] in ('storage_budget_reached', 'local_disk_low', 'commit_error', 'error', 'config_error'):
                raise RuntimeError(f"Acquisition halted safely: {jid}: {r['status']}: {r['error']}")
        print('Round added:', f'{round_new:,}', 'proxy tokens', flush=True)
        if time.monotonic() >= SESSION_DEADLINE:
            break
        if round_new == 0:
            print('STOP: no progress. This is NOT proof of completed quotas; inspect last_job_reports.json.')
            break
        if all(remaining[j['id']] <= 0 or j['id'] in exhausted_jobs for j in JOBS):
            print('STOP: remaining quotas belong to exhausted sources. Manual quota decision required.')
            break

    _diag_stop.set()
    diag_event('acquisition_finished')
